# Growing MLP: добавление групп с заморозкой

$$f(x)=\sin(1.7x)+0.35\sin(5.3x)+0.15x^2-0.3x, \qquad x\in[-3,3].$$

Один скрытый ReLU-слой. `group_sizes` задаёт любое количество групп: первая — начальный слой, остальные добавляются по очереди. По умолчанию группы **1, 1, 2, 4, 8, 16, 32, 64** дают ширину **1 → 2 → 4 → 8 → 16 → 32 → 64 → 128**. На каждом этапе обучается только новая группа, по 200 шагов Adam, всего 1600. После каждого этапа её входящие веса, bias и выходные связи замораживаются навсегда. Общий выходной bias обучается только на первом этапе.

Новая случайно инициализированная группа учится на остаточной ошибке замороженной части. Последняя группа в конце тоже замораживается.

Лосс сверху, целевая функция и текущий фит снизу. Графики Plotly запрашивают свежую модель после завершения предыдущего кадра, не чаще `target_fps`. Промежуточные состояния пропускаются; обучение работает в фоне без пауз.

In [1]:
from pathlib import Path
import sys

project_root = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "notebooks" / "_shared").is_dir()
)
shared_parent = str(project_root / "notebooks")
if shared_parent not in sys.path:
    sys.path.insert(0, shared_parent)

import torch
from _shared import Config, Experiment
from _shared.visualization import train_live

torch.set_num_threads(1)

In [2]:
config = Config(
    seed=42,
    growth_seed=123,
    group_sizes=(1, 1, 2, 4, 8, 16, 32, 64),
    steps_per_stage=200,
    learning_rate=0.01,
    n_train=256,
    n_grid=601,
    x_min=-3.0,
    x_max=3.0,
)
target_fps = 10  # Максимальная частота отрисовки, обучение без пауз.

In [3]:
experiment = Experiment(config)
await train_live(experiment, target_fps=target_fps)